# Análise da calibração das bombas peristálticas

Bancada Litel / UFJF — *Painel de Bombas*. Este notebook documenta **como** cada
bomba foi calibrada, recalcula vazão e coeficiente a partir dos cronômetros e
traça as curvas $Q(\mathrm{PWM})$ usadas no aplicativo.

Fluido: **água**. Volume lido visualmente na escala do **frasco B** (direita).
Uma bomba por vez, nos dois sentidos de rotação. Zona morta $\mathrm{PWM}_0$
obtida por tentativa do menor PWM em que o motor gira de forma contínua; na
maioria dos casos o mesmo valor foi adotado para direto e reverso.


## 1. Bancada e leitura de volume

Dois frascos graduados de **200 ml**, escala útil de **50 ml a 200 ml**:

| Frasco | Posição | Papel |
|---|---|---|
| **A** | esquerda | reservatório / destino da outra mangueira |
| **B** | direita | **frasco observado** — o volume deslocado é lido nesta escala |

Cada bomba peristáltica tem duas mangueiras. Uma vai a A, a outra a B. O sentido
*direto* empurra o líquido num sentido; o *reverso* inverte o fluxo. Em ambos os
casos o operador cronometra o deslocamento de um volume fixo **visto em B**
(aumento ou diminuição do menisco, conforme o sentido).

A leitura é visual, sobre graduação de 50 em 50 ml (e subdivisões do frasco).
Isso limita a resolução do volume — ver seção de incerteza.

Não há sensor de fluxo. A vazão do painel é **estimada** pelo modelo abaixo.


## 2. Modelo usado no painel

Abaixo da zona morta o motor não vence o atrito / a pressão da mangueira:

$$
Q(\mathrm{PWM}) =
\begin{cases}
0 & \text{se } \mathrm{PWM} < \mathrm{PWM}_0 \\
a\,(\mathrm{PWM} - \mathrm{PWM}_0) & \text{se } \mathrm{PWM} \ge \mathrm{PWM}_0
\end{cases}
$$

Unidades neste notebook: $\mathrm{PWM}$ e $\mathrm{PWM}_0$ em **%**; $Q$ em
**ml/min**; $a$ em **ml/min por ponto percentual**.

Com um ensaio de **volume fixo** $V$ cronometrado em $t$:

$$
Q = \frac{V}{t}
\qquad
a = \frac{Q}{\mathrm{PWM}_\text{regime} - \mathrm{PWM}_0}
$$

($t$ convertido para minutos.) Cada par bomba + sentido, hoje, tem **um ponto
operacional** (tipicamente 150 ml @ 90 %) mais o $\mathrm{PWM}_0$ empírico. A
“curva” é portanto o modelo linear por partes ancorado nesse ponto — não um
ajuste com vários PWM. Novas linhas no CSV com outros PWM passam a permitir um
ajuste de $a$ por regressão (célula no final).


## 3. Metodologia — a mesma, bomba a bomba

Calibrar **uma bomba por vez**. Não ligar as outras durante o ensaio.

### 3.1 Zona morta $\mathrm{PWM}_0$

1. Mangueiras já conectadas a A e B, com líquido (sem ar na linha, se possível).
2. Subir o PWM em degraus até o rotor **começar a girar de forma contínua**.
3. Anotar esse PWM como $\mathrm{PWM}_0$ daquela bomba.
4. Repetir no outro sentido. Se a diferença for pequena, **adotar o mesmo
   $\mathrm{PWM}_0$ nos dois sentidos** (foi o caso da maioria das bombas).

É uma aproximação de bancada, não um limiar de estol medido com tacômetro.

### 3.2 Ensaio de vazão (volume fixo)

1. Escolher PWM de regime (nesta campanha: **90 %**) e tempo de estabilização
   (~1 s no painel — espera o regime após o transitório).
2. Escolher o sentido (direto ou reverso).
3. Cronometrar até o volume em **B** deslocar o alvo (nesta campanha: **150 ml**).
4. Registrar $V$, $t$, PWM, sentido, $\mathrm{PWM}_0$.
5. Inverter o sentido e repetir.

### 3.3 Ensaio alternativo (tempo fixo)

O painel também permite PWM e tempo fixos, informando $V$ depois. O cálculo de
$Q$ e $a$ é o mesmo. Os dados desta pasta são todos de **volume fixo**.

### 3.4 O que é específico de cada bomba

Não muda o método. Mudam apenas:

- o $\mathrm{PWM}_0$ empírico (tubo, desgaste, ponte H, montagem);
- o tempo para deslocar 150 ml @ 90 % (logo $Q$ e $a$);
- a diferença direto × reverso (assimetria da cabeça / das mangueiras).


## 4. Ensaios desta campanha

Água; frascos 200 ml; observação em B; volume fixo 150 ml; PWM de regime 90 %.
P04–P06 em 22/09/2026 (app instalado). P01–P03 em 23/09/2026 (sessão de
desenvolvimento). $\mathrm{PWM}_0$ igual nos dois sentidos em todas as bombas.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 150,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.35,
        "font.size": 10,
    }
)

HERE = Path.cwd()
CSV = HERE / "ensaios_calibracao.csv"
if not CSV.exists():
    CSV = HERE / "scripts" / "ensaios_calibracao.csv"
FIG = CSV.parent / "figuras"
FIG.mkdir(exist_ok=True)

ensaios = pd.read_csv(CSV, parse_dates=["data_hora"])
ensaios


## 5. Recalcular $Q$ e $a$

Confere os coeficientes gravados no painel: $Q = 60V/t$ (ml/min) e
$a = Q/(\mathrm{PWM}-\mathrm{PWM}_0)$.


In [ ]:
def vazao_ml_min(volume_ml: pd.Series, tempo_s: pd.Series) -> pd.Series:
    return 60.0 * volume_ml / tempo_s


def coeficiente_a(q_ml_min: pd.Series, pwm: pd.Series, pwm0: pd.Series) -> pd.Series:
    span = pwm - pwm0
    if (span <= 0).any():
        raise ValueError("PWM de regime precisa ser maior que PWM0")
    return q_ml_min / span


def q_modelo(pwm, a, pwm0):
    pwm = np.asarray(pwm, dtype=float)
    return np.where(pwm < pwm0, 0.0, a * (pwm - pwm0))


calc = ensaios.copy()
calc["Q_ml_min"] = vazao_ml_min(calc["volume_ml"], calc["tempo_s"])
calc["a"] = coeficiente_a(calc["Q_ml_min"], calc["pwm_regime"], calc["pwm0"])
calc["Q_em_100"] = q_modelo(100, calc["a"], calc["pwm0"])

resumo = (
    calc[
        [
            "bomba",
            "sentido",
            "data_hora",
            "pwm0",
            "pwm_regime",
            "volume_ml",
            "tempo_s",
            "Q_ml_min",
            "a",
        ]
    ]
    .sort_values(["bomba", "sentido"])
    .reset_index(drop=True)
)
resumo


## 6. Tabela por bomba

Uma linha por bomba: $\mathrm{PWM}_0$, $a$ e $Q$ nos dois sentidos, e o
desvio relativo direto × reverso em 90 %.


In [ ]:
def pivot_sentido(df: pd.DataFrame) -> pd.DataFrame:
    linhas = []
    for bomba, grp in df.groupby("bomba"):
        d = grp.set_index("sentido")
        direto = d.loc["direto"]
        reverso = d.loc["reverso"]
        q_rel = 100.0 * (direto.Q_ml_min - reverso.Q_ml_min) / reverso.Q_ml_min
        linhas.append(
            {
                "bomba": int(bomba),
                "PWM0_%": direto.pwm0,
                "t_direto_s": direto.tempo_s,
                "t_reverso_s": reverso.tempo_s,
                "Q_direto": direto.Q_ml_min,
                "Q_reverso": reverso.Q_ml_min,
                "a_direto": direto.a,
                "a_reverso": reverso.a,
                "delta_Q_%": q_rel,
            }
        )
    return pd.DataFrame(linhas).set_index("bomba")


por_bomba = pivot_sentido(calc)
por_bomba


## 7. Curvas de calibração — uma figura por bomba

Cada painel mostra $Q=0$ até $\mathrm{PWM}_0$ e a reta que passa pelo ensaio
de 90 %. O ponto marcado é o volume fixo cronometrado. Direto e reverso
compartilham o $\mathrm{PWM}_0$ desta campanha, mas têm $a$ próprios.


In [ ]:
COR = {"direto": "#1d4ed8", "reverso": "#c2410c"}
pwm_eixo = np.linspace(0, 100, 401)

fig, eixos = plt.subplots(2, 3, figsize=(11.2, 6.8), sharex=True, sharey=True)
eixos = eixos.ravel()

for i, bomba in enumerate(range(1, 7)):
    ax = eixos[i]
    grp = calc[calc["bomba"] == bomba]
    pwm0 = float(grp["pwm0"].iloc[0])
    ax.axvspan(0, pwm0, color="#94a3b8", alpha=0.18, zorder=0)
    ax.axvline(pwm0, color="#64748b", ls="--", lw=1, label=f"PWM₀ = {pwm0:.0f} %")
    for _, row in grp.iterrows():
        cor = COR[row.sentido]
        ax.plot(
            pwm_eixo,
            q_modelo(pwm_eixo, row.a, row.pwm0),
            color=cor,
            lw=2,
            label=f"{row.sentido}  a = {row.a:.3f}",
        )
        ax.scatter(
            [row.pwm_regime],
            [row.Q_ml_min],
            color=cor,
            s=36,
            zorder=3,
            edgecolors="white",
            linewidths=0.6,
        )
    ax.set_title(f"P0{bomba}")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, None)
    ax.legend(fontsize=7.5, loc="upper left", frameon=False)

for ax in eixos[3:]:
    ax.set_xlabel("PWM (%)")
for ax in (eixos[0], eixos[3]):
    ax.set_ylabel("Q (ml/min)")

fig.suptitle("Curvas de calibração — água, volume fixo 150 ml lido em B", y=1.01)
fig.tight_layout()
fig.savefig(FIG / "curvas_por_bomba.png", bbox_inches="tight")
plt.show()


## 8. As seis bombas no mesmo eixo

Compara capacidade em direto e em reverso. A faixa cinza de cada curva começa
no respectivo $\mathrm{PWM}_0$.


In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(11.2, 4.4), sharey=True)
cmap = plt.cm.tab10

for ax, sentido in zip(eixos, ("direto", "reverso")):
    subset = calc[calc["sentido"] == sentido]
    for _, row in subset.iterrows():
        cor = cmap(int(row.bomba) - 1)
        ax.plot(
            pwm_eixo,
            q_modelo(pwm_eixo, row.a, row.pwm0),
            color=cor,
            lw=2,
            label=f"P0{int(row.bomba)}  PWM₀={row.pwm0:.0f}%",
        )
        ax.scatter([row.pwm_regime], [row.Q_ml_min], color=cor, s=28, zorder=3)
    ax.set_title(sentido.capitalize())
    ax.set_xlabel("PWM (%)")
    ax.set_xlim(0, 100)
    ax.set_ylim(0, None)
    ax.legend(fontsize=8, frameon=False, loc="upper left")

eixos[0].set_ylabel("Q (ml/min)")
fig.suptitle("Comparativo das seis bombas", y=1.02)
fig.tight_layout()
fig.savefig(FIG / "comparativo_seis_bombas.png", bbox_inches="tight")
plt.show()


## 9. Zona morta e assimetria direto / reverso


In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(11.2, 3.6))
x = por_bomba.index.astype(int)

eixos[0].bar(x, por_bomba["PWM0_%"], color="#334155", width=0.65)
eixos[0].set_title("PWM₀ adotado")
eixos[0].set_xlabel("Bomba")
eixos[0].set_ylabel("%")
eixos[0].set_xticks(x)
eixos[0].set_xticklabels([f"P0{i}" for i in x])

largura = 0.36
eixos[1].bar(x - largura / 2, por_bomba["a_direto"], width=largura, color=COR["direto"], label="direto")
eixos[1].bar(x + largura / 2, por_bomba["a_reverso"], width=largura, color=COR["reverso"], label="reverso")
eixos[1].set_title("Coeficiente a")
eixos[1].set_xlabel("Bomba")
eixos[1].set_ylabel("ml/min / %")
eixos[1].set_xticks(x)
eixos[1].set_xticklabels([f"P0{i}" for i in x])
eixos[1].legend(frameon=False, fontsize=8)

cores_delta = ["#1d4ed8" if v >= 0 else "#c2410c" for v in por_bomba["delta_Q_%"]]
eixos[2].bar(x, por_bomba["delta_Q_%"], color=cores_delta, width=0.65)
eixos[2].axhline(0, color="#64748b", lw=1)
eixos[2].set_title("Q direto − Q reverso @ 90 %")
eixos[2].set_xlabel("Bomba")
eixos[2].set_ylabel("%")
eixos[2].set_xticks(x)
eixos[2].set_xticklabels([f"P0{i}" for i in x])

fig.tight_layout()
fig.savefig(FIG / "pwm0_e_assimetria.png", bbox_inches="tight")
plt.show()

print("PWM0: min", por_bomba["PWM0_%"].min(), "max", por_bomba["PWM0_%"].max())
print("Maior assimetria |ΔQ|:", por_bomba["delta_Q_%"].abs().idxmax(),
      f"{por_bomba['delta_Q_%'].abs().max():.1f} %")


## 10. Incerteza da leitura visual

Os frascos vão de 50 ml a 200 ml. O alvo de 150 ml cabe na escala (por exemplo
50 → 200 ml, ou 0 efetivo da graduação até 150). O menisco é lido a olho.

Se a incerteza de volume for da ordem de $\delta V \approx 5\,\mathrm{ml}$ e a
do cronômetro $\delta t \approx 0{,}5\,\mathrm{s}$, a vazão propaga

$$
\frac{\delta Q}{Q} \approx \sqrt{\left(\frac{\delta V}{V}\right)^2 + \left(\frac{\delta t}{t}\right)^2}.
$$

Com $V=150\,\mathrm{ml}$ e $t\sim 100\,\mathrm{s}$ isso dá cerca de **3–4 %**
em $Q$ (e portanto em $a$), dominado pelo volume. $\mathrm{PWM}_0$ é ainda mais
grosseiro: um degrau de PWM na busca do “começou a girar”.

Por isso o modelo é operacional para o painel, não um certificado metrológico.
Repetir o ensaio no mesmo PWM e no mesmo sentido é o próximo passo para
estimar repetibilidade (hoje há **uma** corrida por bomba e sentido).


In [ ]:
delta_V = 5.0
delta_t = 0.5
inc = resumo.copy()
inc["rel_V_%"] = 100 * delta_V / inc["volume_ml"]
inc["rel_t_%"] = 100 * delta_t / inc["tempo_s"]
inc["rel_Q_%"] = np.sqrt(inc["rel_V_%"] ** 2 + inc["rel_t_%"] ** 2)
inc["Q_lo"] = inc["Q_ml_min"] * (1 - inc["rel_Q_%"] / 100)
inc["Q_hi"] = inc["Q_ml_min"] * (1 + inc["rel_Q_%"] / 100)
inc[["bomba", "sentido", "Q_ml_min", "rel_Q_%", "Q_lo", "Q_hi"]]


## 11. Incluir novos ensaios

Acrescente linhas em `ensaios_calibracao.csv` (mesmo cabeçalho). Se a mesma
bomba e o mesmo sentido tiverem **vários PWM de regime**, a célula abaixo
ajusta $a$ por mínimos quadrados em $Q$ versus $(\mathrm{PWM}-\mathrm{PWM}_0)$,
forçando passagem na origem do modelo (zona morta já descontada).


In [ ]:
def ajustar_a(grupo: pd.DataFrame) -> pd.Series:
    x = (grupo["pwm_regime"] - grupo["pwm0"]).to_numpy(float)
    y = grupo["Q_ml_min"].to_numpy(float)
    a_fit = float(np.dot(x, y) / np.dot(x, x))
    return pd.Series(
        {
            "n": int(len(grupo)),
            "pwm0": float(grupo["pwm0"].iloc[0]),
            "a_ensaio_unico": float(grupo["a"].iloc[-1]),
            "a_ajuste": a_fit,
        }
    )


ajuste = (
    calc.groupby(["bomba", "sentido"], sort=True)
    .apply(ajustar_a, include_groups=False)
    .reset_index()
)
ajuste


## 12. Arquivos gerados

Ao rodar o notebook, as figuras vão para `scripts/figuras/`:

- `curvas_por_bomba.png`
- `comparativo_seis_bombas.png`
- `pwm0_e_assimetria.png`

Dependências: `pip install -r scripts/requirements.txt`. Abra o notebook a
partir de `scripts/` para o CSV ser encontrado no diretório atual.
